# MicroStrategy REST Admin

## Load libraries

In [1]:
import json
from mstrio.object_management import folder
from mstrio.api import metrics,browsing,filters,attributes,transformations
import pandas as pd
from time import sleep
import shutil
import os
from mstrio.connection import Connection
all_comp_obj_d_l=[]

In [2]:
from mstrio.connection import Connection
with open('..\\config\\user_d.json', 'r') as openfile:
    user_d = json.load(openfile)

conn_params =  user_d["conn_params"]
conn = Connection(**conn_params)
conn.headers['Content-type'] = "application/json"
project_id="B7CA92F04B9FAE8D941C3E9B7E0CD754"
conn.select_project(project_id)

Connection to Strategy One Intelligence Server has been established.
No project selected.


## Endpoints

In [4]:
def get_obj_details(conn,object_id):
    body_d={
      "projectIdAndObjectIds": [
        {
          "projectId": conn.project_id,
          "objectIds": [
            object_id
          ]
        }
      ]
    }
    body_j=json.dumps(body_d)
    url=f'{conn.base_url}/api/searches/objects?includeAncestors=false&showNavigationPath=false'
    obj_det_d=conn.post(url,data=body_j).json()["result"][0]
    return obj_det_d
    
def get_child_objects(conn,object_id):
    limit=500
    dpn_child_d_l=[]
    object_det_d=get_obj_details(conn,object_id)
    
    try:
        search_instance_resp=browsing.store_search_instance(connection=conn,project_id=conn.project_id,uses_object=object_id+";"+ str(object_det_d["type"]))
        dpn_count=search_instance_resp.json()["totalItems"]
        if dpn_count>0:
            full_result_d_l=[]
            fetched_items_count=0
            while fetched_items_count <dpn_count:
                search_d_l=browsing.get_search_results(
                connection= conn,
                search_id=search_instance_resp.json()["id"],
                project_id=conn.project_id,
                offset=fetched_items_count,
                limit=limit).json()
                full_result_d_l.extend(search_d_l)
                fetched_items_count+=limit
                
                print(dpn_count)
    except Exception as e:
        print(e)
    for o in full_result_d_l:
        dpn_child_d={}
        dpn_child_d["id"]=object_det_d["id"]
        dpn_child_d["type"]=object_det_d["type"]
        dpn_child_d["subtype"]=object_det_d["subtype"]
        dpn_child_d["name"]=o["id"]
        dpn_child_d["child_dpn_id"]=o["id"]
        dpn_child_d["child_dpn_type"]=o["type"]
        dpn_child_d["child_dpn_subtype"]=o["subtype"]
        dpn_child_d["child_dpn_name"]=o["name"]
        dpn_child_d_l.append(dpn_child_d.copy())

    return dpn_child_d_l

In [5]:
#metric
metric_d_l=[]
metric_id="4C051DB611D3E877C000B3B2D86C964F"
metric_l=[metric_id]
for metric_id in metric_l:
    metric_def=metrics.get_metric(
        connection=conn,
        id=metric_id,
        changeset_id=None,
        show_expression_as="tokens",
        show_filter_tokens=False
    )
    metric_def=metric_def.json()
    metric_d={}
    metric_d["project_id"]=conn.project_id
    metric_d["id"]=metric_id
    metric_d["name"]=metric_def["name"]
    metric_d["text"]=metric_def["expression"]["text"]
    metric_d["expression"]=metric_def["expression"]
    metric_d["transformations_l"]=None
    metric_d["commplexity"]=1
    metric_d_l.append(metric_d.copy())
    all_comp_obj_d_l.append(get_child_objects(conn,object_id=metric_id))

metric_d_l


390


[{'project_id': 'B7CA92F04B9FAE8D941C3E9B7E0CD754',
  'id': '4C051DB611D3E877C000B3B2D86C964F',
  'name': 'Profit',
  'text': 'Sum(Profit)',
  'expression': {'text': 'Sum(Profit)',
   'tokens': [{'level': 'resolved',
     'state': 'initial',
     'value': 'Sum',
     'type': 'function',
     'target': {'dateCreated': '2001-01-02T20:47:35.000Z',
      'dateModified': '2025-05-27T17:03:09.851Z',
      'versionId': '0463E7DB404DA9839CF29D82B552B31B',
      'acg': 255,
      'primaryLocale': 'en-US',
      'objectId': '8107C31BDD9911D3B98100C04F2233EA',
      'subType': 'function',
      'name': 'Sum',
      'description': 'Returns the sum of all values in the ValueList.  This is a group-value function.'}},
    {'level': 'resolved',
     'state': 'initial',
     'value': '<',
     'type': 'character'},
    {'level': 'resolved',
     'state': 'initial',
     'value': 'UseLookupForAttributes',
     'type': 'identifier'},
    {'level': 'resolved',
     'state': 'initial',
     'value': '=',
 

In [6]:
#filter
filter_d_l=[]
filter_l=["0768BE624F8FD86D6AD7978A94EC5844","8827905B11D3EB22C000B4B2D86C964F","E7773E5B4ECCCA9A22B23CB3367F21DB"]
for filter_id in filter_l:
    filter_def=filters.get_filter(connection=conn, id=filter_id, project_id=project_id).json()
    filter_d={}
    filter_d["project_id"]=conn.project_id
    filter_d["id"]=filter_id
    filter_d["name"]=filter_def["name"]
    filter_d["text"]=filter_def["qualification"]["text"]
    filter_d["filter_type"]=filter_def["qualification"]["tree"]["type"]
    filter_d["commplexity"]=1
    filter_d["qualification"]=filter_def["qualification"]
    filter_d_l.append(filter_d.copy())
    all_comp_obj_d_l.append(get_child_objects(conn,object_id=filter_id))
filter_d_l

1
1
2


[{'project_id': 'B7CA92F04B9FAE8D941C3E9B7E0CD754',
  'id': '0768BE624F8FD86D6AD7978A94EC5844',
  'name': 'TV & Video',
  'text': "Subcategory = TV's, Video Equipment",
  'filter_type': 'predicate_element_list',
  'commplexity': 1,
  'qualification': {'text': "Subcategory = TV's, Video Equipment",
   'tree': {'type': 'predicate_element_list',
    'predicateId': 'B3043B7ED00B4A228959C801BB3D5862',
    'predicateTree': {'attribute': {'objectId': '8D679D4F11D3E4981000E787EC6DE8A4',
      'subType': 'attribute',
      'name': 'Subcategory'},
     'elements': [{'display': "TV's", 'elementId': 'h25'},
      {'display': 'Video Equipment', 'elementId': 'h26'}],
     'function': 'in'}}}},
 {'project_id': 'B7CA92F04B9FAE8D941C3E9B7E0CD754',
  'id': '8827905B11D3EB22C000B4B2D86C964F',
  'name': 'by age',
  'text': '(Customer where ({Customer Age} (ID) Between {Customers segmentation by age} and {Customer segmentation by age}) Relate by LU_CUSTOMER)',
  'filter_type': 'predicate_relationship',
  '

In [11]:
#attributes
attribute_d_l=[]   
att_parent_child_d_l=[] 
attribute_l=["8D679D3C11D3E4981000E787EC6DE8A4","54BABD9E11D59D57C000B28A4CC5F24F"]
attribute_def=attributes.get_attribute(connection=conn,id=attribute_l[0],show_expression_as="tokens").json()

for att in attribute_l:
    attribute_d={}
    attribute_d["project_id"]=conn.project_id
    attribute_def=attributes.get_attribute(connection=conn,id=att,show_expression_as="tokens").json()
    attribute_d["id"]=attribute_def["id"]
    attribute_d["name"]=attribute_def["name"]
    #attribute_d["text"]=attribute_def["expression"]["text"]
    attribute_d["def"]=attribute_def
    attribute_d_l.append(attribute_d.copy())


for r in attribute_def["relationships"]:
  
    att_parent_child_d={}

    if attribute_def["id"]!=r["parent"]["objectId"]:

        att_parent_child_d["rel_attribute_id"]=r["parent"]["objectId"]
        att_parent_child_d["rel_attribute_name"]=r["parent"]["name"]
        att_parent_child_d["rel_table"]=r["relationshipTable"]["name"]
        att_parent_child_d["rel_type"]=r["relationshipType"]
        att_parent_child_d["type"]="parent"
    else:

        att_parent_child_d["rel_attribute_id"]=r["child"]["objectId"]
        att_parent_child_d["rel_attribute_name"]=r["child"]["name"]
        att_parent_child_d["rel_table"]=r["relationshipTable"]["name"]
        att_parent_child_d["rel_type"]=r["relationshipType"]
        att_parent_child_d["type"]="child"
    att_parent_child_d_l.append(att_parent_child_d.copy())
att_parent_child_d_l

[{'rel_attribute_id': '8D679D3C11D3E4981000E787EC6DE8A4',
  'rel_attribute_name': 'Customer',
  'rel_table': 'LU_CUSTOMER',
  'rel_type': 'one_to_many',
  'type': 'child'},
 {'rel_attribute_id': '5B81D4D541A551CCC0535EA54B88B611',
  'rel_attribute_name': 'Customer Latitude',
  'rel_table': 'LU_CUSTOMER',
  'rel_type': 'one_to_one',
  'type': 'parent'},
 {'rel_attribute_id': 'A85A9D5C4F64009D39EA96812FFFF271',
  'rel_attribute_name': 'Customer Longitude',
  'rel_table': 'LU_CUSTOMER',
  'rel_type': 'one_to_one',
  'type': 'parent'}]

In [ ]:
#facts

In [55]:
#transformations
transformation_d_l=[]   
transformation_l=["6CB9ABFD11D3E4F11000E887EC6DE8A4"]
for t in transformation_l:
    transformation_d={}
    trans_def=transformations.get_transformation(connection=conn,id=t).json()
    transformation_d["id"]=trans_def["id"]
    transformation_d["name"]=trans_def["name"]
    transformation_d["mapping_type"]=trans_def["mappingType"]
    transformation_d["def"]=trans_def
    transformation_d_l.append(transformation_d.copy())


transformation_d_l

[{'id': '6CB9ABFD11D3E4F11000E887EC6DE8A4',
  'name': 'Month to Date',
  'mapping_type': 'many_to_many',
  'def': {'attributes': [{'id': '96ED427011D5B117C000E78A4CC5F24F',
     'baseAttribute': {'objectId': '96ED3EC811D5B117C000E78A4CC5F24F',
      'subType': 'attribute',
      'name': 'Day'},
     'forms': [{'id': '45C11FA478E745FEA08D781CEA190FE5',
       'name': 'ID',
       'lookupTable': {'objectId': '24C30AD711D5AEC9C000E38A4CC5F24F',
        'subType': 'logical_table',
        'name': 'MTD_DAY'},
       'expression': {'text': 'mtd_day_date'}}]}],
   'mappingType': 'many_to_many',
   'dateCreated': '2001-01-02T20:47:18.000Z',
   'dateModified': '2004-03-17T16:56:48.000Z',
   'versionId': '5614F9FB47F0CE2C250EC29021737A8B',
   'acg': 255,
   'primaryLocale': 'en-US',
   'subType': 'role_transformation',
   'name': 'Month to Date',
   'description': 'Transforms Day to Month to Date',
   'id': '6CB9ABFD11D3E4F11000E887EC6DE8A4'}}]

In [ ]:
#Olap reports
report_d_l=[]
report_l=["5D55D4504907E73F560BFBA95AD22A66","25D40AD444B6D51B333021ADFB219501"]
for report_id in report_l:
    all_comp_obj_d_l.append(get_child_objects(conn,object_id=report_id))

390


In [61]:
#OlapCubes
olap_cube_l=["4E3A75BB43FF442926F3BDA90528CE47"]
report_l=["5D55D4504907E73F560BFBA95AD22A66","25D40AD444B6D51B333021ADFB219501"]
for cube_id in olap_cube_l:
    all_comp_obj_d_l.append(get_child_objects(conn,object_id=cube_id))

UnboundLocalError: cannot access local variable 'full_result_d_l' where it is not associated with a value

In [ ]:
#MTDI Cubes